# AIS Spoofing Scenarios: Dutch North Sea Waters

This notebook demonstrates that **structurally valid and contextually plausible AIS messages** can be manually constructed using the validated `pyais` encoder and knowledge of real traffic distributions from the Exploratory Data Analysis (EDA).

Each scenario defines a threat narrative, constructs realistic AIS field values, encodes them into valid NMEA 0183 sentences, verifies the result through round-trip decoding, and provides a geographic visualization of where the spoofed vessel would appear.

All scenarios are situated in or near the **Netherlands' North Sea waters** and use:
- Valid MMSI prefixes for the claimed flag state
- Coordinates verified to be in water (not on land)
- Field values within ITU-R M.1371 valid ranges
- Values informed by EDA baseline distributions
- Structurally valid NMEA 0183 sentences produced by the `pyais` encoder (validated with 28/28 tests passing)

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
from pyais import decode
from pyais.encode import encode_dict

print("Dependencies loaded successfully.")

Dependencies loaded successfully.


In [11]:
def encode_and_verify(vessel_data, label):
    """Encode a vessel dictionary, decode it back, print both, verify round-trip."""
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}")

    print(f"\n  INPUT FIELD VALUES:")
    print(f"    MMSI:       {vessel_data['mmsi']}")
    print(f"    Lat:        {vessel_data['lat']}")
    print(f"    Lon:        {vessel_data['lon']}")
    print(f"    SOG:        {vessel_data['speed']} knots")
    print(f"    COG:        {vessel_data['course']} degrees")
    print(f"    Heading:    {vessel_data['heading']} degrees")
    print(f"    ROT:        {vessel_data.get('turn', 0.0)}")
    print(f"    Nav Status: {vessel_data['status']}")

    encoded = encode_dict(vessel_data)
    print(f"\n  ENCODED NMEA SENTENCE:")
    print(f"    {encoded[0]}")

    decoded = decode(*encoded).asdict()
    print(f"\n  DECODED BACK:")
    print(f"    MMSI:       {decoded['mmsi']}")
    print(f"    Lat:        {decoded['lat']}")
    print(f"    Lon:        {decoded['lon']}")
    print(f"    SOG:        {decoded['speed']} knots")
    print(f"    COG:        {decoded['course']} degrees")
    print(f"    Heading:    {decoded['heading']} degrees")
    print(f"    ROT:        {decoded['turn']}")
    print(f"    Nav Status: {decoded['status']}")

    # Verify round-trip
    checks = {
        'MMSI': str(decoded['mmsi']) == vessel_data['mmsi'],
        'SOG': decoded['speed'] == vessel_data['speed'],
        'COG': decoded['course'] == vessel_data['course'],
        'Heading': decoded['heading'] == vessel_data['heading'],
        'Lat': abs(decoded['lat'] - vessel_data['lat']) < 0.001,
        'Lon': abs(decoded['lon'] - vessel_data['lon']) < 0.001,
    }
    all_ok = all(checks.values())
    print(f"\n  ROUND-TRIP: {'PASS' if all_ok else 'FAIL'}")
    if not all_ok:
        for field, ok in checks.items():
            if not ok:
                print(f"    {field}: MISMATCH")

    return encoded, decoded

---
## Scenario 1: Ghost Vessel Blocking the Nieuwe Waterweg Entrance

### Threat Narrative

The Port of Rotterdam is Europe's largest seaport, handling approximately 30,000 sea-going vessel visits per year. All inbound and outbound deep-draught traffic must transit the **Nieuwe Waterweg**, the sole direct ship canal connecting Rotterdam to the North Sea. At the western end of this canal, near Hook of Holland, the **Maeslantkering** storm surge barrier marks the narrowest navigable section, a chokepoint just 480 to 675 metres wide, dredged to 14.5–16 metres depth.

In this scenario, an attacker injects a **single spoofed AIS position report** representing a large vessel broadcasting **navigational status 2 (not under command)** at **SOG 0.3 knots**, positioned diagonally across the Nieuwe Waterweg directly at the Maeslantkering barrier passage. The vessel's heading is set to 25° (NNE), oriented perpendicular to the channel that runs at approximately 110°/290° at this point, indicating the vessel has swung sideways across the fairway. This is the worst-case orientation for a drifting vessel in a narrow canal: it blocks the full navigable width.

### Why This Is Plausible

| Aspect | Justification |
|--------|---------------|
| **Location** | 51.955°N, 4.163°E —> Nieuwe Waterweg at the Maeslantkering passage, the narrowest point of Rotterdam's sea access |
| **MMSI** | 244810001 —> valid Dutch prefix (244), fictitious vessel number |
| **SOG = 0.3** | Near-stationary but drifting slightly with tidal current; realistic for a vessel without propulsion in the tidally influenced Nieuwe Waterweg. 61.3% of cleaned AIS records in our dataset have SOG = 0, making low-speed values common |
| **Nav status = 2** | "Not under command" —> a valid ITU-R M.1371 navigational status code indicating loss of propulsion or steering |
| **Heading 25° ≠ COG 290°** | The vessel's bow points NNE while drifting WNW with the outgoing tidal current —> it has swung perpendicular to the ~110°/290° channel axis, blocking the full navigable width |
| **ROT = -1.5** | Slow counter-clockwise rotation, consistent with asymmetric current forces on a drifting vessel |
| **Channel width** | At 480–675 m, a single large vessel oriented diagonally can obstruct the full navigable width |


### Potential Impact

- Vessel Traffic Services (VTS) must treat the AIS report as genuine, the protocol has no authentication mechanism
- All inbound **and** outbound traffic to the Port of Rotterdam would be halted until the obstruction is verified
- Tugs and coast guard vessels may be dispatched to assist a vessel that does not exist
- The Maeslantkering barrier operators may need to assess whether the "drifting" vessel poses a risk to the barrier structure itself
- Even minutes of disruption at this chokepoint cascades into supply chain delays for Europe's largest port

In [12]:
# Scenario 1: Spoofed vessel field values
scenario_1 = {
    'type': 1,
    'mmsi': '244810001',       # Dutch MMSI prefix (244), fictitious vessel
    'status': 2,               # Not under command
    'turn': -1.5,              # Very slow counter-clockwise drift (tidal current)
    'speed': 0.3,              # Near-stationary, slight drift with current
    'accuracy': 1,             # High accuracy GPS
    'lon': 4.1628,             # Nieuwe Waterweg at Maeslantkering passage
    'lat': 51.9552,            # Narrowest point of Rotterdam sea access
    'course': 290.0,           # Drifting west-northwest with outgoing current
    'heading': 25,             # Vessel swung NNE, perpendicular to the channel
    'second': 15,
    'maneuver': 0,
    'raim': 0,
}

encoded_1, decoded_1 = encode_and_verify(
    scenario_1, "Scenario 1: Ghost vessel blocking Nieuwe Waterweg at Maeslantkering"
)


  Scenario 1: Ghost vessel blocking Nieuwe Waterweg at Maeslantkering

  INPUT FIELD VALUES:
    MMSI:       244810001
    Lat:        51.9552
    Lon:        4.1628
    SOG:        0.3 knots
    COG:        290.0 degrees
    Heading:    25 degrees
    ROT:        -1.5
    Nav Status: 2

  ENCODED NMEA SENTENCE:
    !AIVDO,1,1,,A,13aN14BvP3PC3TPMfb0;E0jN0000,0*6C

  DECODED BACK:
    MMSI:       244810001
    Lat:        51.9552
    Lon:        4.1628
    SOG:        0.3 knots
    COG:        290.0 degrees
    Heading:    25 degrees
    ROT:        -2.0
    Nav Status: 2

  ROUND-TRIP: PASS


### Geographic Context

![Scenario 1 — Overview](../data/processed/Scenario1_2026-04-02_180511.png)

![Scenario 1 — Zoomed](../data/processed/Scenario1_zoomed_2026-04-02_180511.png)

---
## Scenario 2: Collision Threat to Autonomous Service Vessel near Hollandse Kust Zuid / Luchterduinen Offshore Wind Farm

### Threat Narrative

The windenergy are **Hollandse Kust Zuid (HKZ)**, including the **Luchterduinen Offshore Wind Farm**, is one of the largest offshore wind farms in the world, located 18–36 km off the Dutch coast between Scheveningen and Zandvoort, with 139 turbines generating 1.5 GW. The wind farm is serviced out of the port of IJmuiden using crew transfer vessels and increasingly **autonomous or semi-autonomous service vessels** for maintenance, inspection, and cargo delivery. These vessels rely heavily on AIS as a primary input for collision avoidance algorithms alongside radar, LiDAR, and camera systems.

A passageway for vessels under 46 metres has been established through the wind farm zone, with **operational AIS mandatory** for all vessels using it. This creates a confined navigation environment where evasive options are limited by the 150-metre minimum distance requirement from turbine foundations.

In this scenario, an attacker injects a spoofed AIS message representing a **fast-moving vessel on a direct collision course** with an autonomous service vessel operating inside the HKZ wind farm passageway. The spoofed vessel appears to be approaching at 19.5 knots from the southwest, a speed and bearing that would trigger the autonomous vessel's COLREGS-based collision avoidance system. Crucially, the spoofed vessel has **no corresponding radar return**, since it does not physically exist. An autonomous vessel that trusts AIS over radar in its sensor fusion algorithm would initiate evasive action, potentially steering toward a turbine foundation or into the path of real traffic in the narrow passageway.

### Why This Is Plausible

| Aspect | Justification |
|--------|---------------|
| **Location** | 52.310°N, 4.100°E —> inside the HKZ wind farm passageway zone, approximately 22 km offshore |
| **MMSI** | 211987654 —> valid German prefix (211), representing foreign commercial traffic transiting Dutch waters |
| **SOG = 19.5** | Realistic speed for a fast supply vessel or small cargo ship; within the valid range observed in the EDA (max 72.2 knots, 75th percentile 4.8 knots for all traffic) |
| **Nav status = 0** | "Under way using engine" —> the most common status in the dataset (58.1% of cleaned records) |
| **COG = 42°** | Heading northeast, directly into the wind farm passageway —> a plausible inbound course from the southwest |
| **Heading ≈ COG** | Heading 40° closely matches COG 42°, indicating stable, intentional navigation —> not a drifting vessel, making it appear as a deliberate approach |
| **Autonomous vessel vulnerability** | MASS rely on sensor fusion where AIS is a primary input; a ghost vessel with no radar return creates a conflicting sensor situation that collision avoidance algorithms may not be trained to handle |

### Potential Impact

- An autonomous vessel's collision avoidance system may initiate emergency evasive action toward turbine foundations or real traffic
- The confined passageway (150 m minimum from turbines) limits evasive options, increasing collision risk with infrastructure
- A human operator at a Shore Control Centre would see conflicting data: AIS shows a vessel, radar shows nothing, creating decision paralysis during critical seconds
- If the attack is timed during poor visibility (fog, night), the absence of a radar return may not be immediately noticed
- Repeated spoofing could erode trust in AIS data within the wind farm zone, degrading overall situational awareness for all operators

In [13]:
# Scenario 2: Spoofed vessel field values
scenario_2 = {
    'type': 1,
    'mmsi': '211987654',       # German MMSI prefix (211), fictitious vessel
    'status': 0,               # Under way using engine
    'turn': 0.0,               # Straight course, deliberate approach
    'speed': 19.5,             # Fast-moving supply/cargo vessel
    'accuracy': 1,             # High accuracy GPS
    'lon': 4.1000,             # Inside Hollandse Kust Zuid wind farm zone
    'lat': 52.3100,            # Approximately 22 km offshore
    'course': 42.0,            # Northeast, into the passageway
    'heading': 40,             # Closely matches COG, stable navigation
    'second': 45,
    'maneuver': 0,
    'raim': 0,
}

encoded_2, decoded_2 = encode_and_verify(
    scenario_2, "Scenario 2: Collision threat to autonomous vessel in HKZ wind farm"
)


  Scenario 2: Collision threat to autonomous vessel in HKZ wind farm

  INPUT FIELD VALUES:
    MMSI:       211987654
    Lat:        52.31
    Lon:        4.1
    SOG:        19.5 knots
    COG:        42.0 degrees
    Heading:    40 degrees
    ROT:        0.0
    Nav Status: 0

  ENCODED NMEA SENTENCE:
    !AIVDO,1,1,,A,13::diP033PBi;0MsaT1a1AJ0000,0*5A

  DECODED BACK:
    MMSI:       211987654
    Lat:        52.31
    Lon:        4.1
    SOG:        19.5 knots
    COG:        42.0 degrees
    Heading:    40 degrees
    ROT:        0.0
    Nav Status: 0

  ROUND-TRIP: PASS


### Geographic Context

![Scenario 2 — Overview](../data/processed/Scenario2.png)

![Scenario 2 — Zoomed](../data/processed/Scenario2_zoomed.png)

---
## Scenario 3: Deceptive Vessel Presence near Borssele Offshore Wind Farm

### Threat Narrative

The **Borssele wind farm zone** is the Netherlands' first large-scale offshore wind installation, located approximately 22–45 km off the coast of Zeeland near the Belgian border, with a combined capacity of over 1.5 GW across five sites. The zone sits near critical subsea infrastructure including export cables connecting to the onshore grid at the Maasvlakte, as well as international subsea telecommunications cables and gas pipelines crossing the southern North Sea.

Following the **sabotage of the Nord Stream 1 and 2 pipelines** in September 2022, the security of subsea infrastructure in the North Sea became a top-priority national security concern. In April 2024, six North Sea nations (Belgium, the Netherlands, Germany, Norway, the United Kingdom, and Denmark) signed a **joint declaration on protecting critical subsea infrastructure**. Suspicious vessel activity near offshore energy installations, particularly by vessels flagged to non-allied states, is now actively monitored and treated as a potential security threat.

In this scenario, an attacker spoofs an AIS message representing a vessel with a **Russian MMSI** operating at low speed near the Borssele wind farm, exhibiting a **loitering pattern** consistent with surveillance or survey activity. The vessel broadcasts nav_status 0 (under way) with slow speed and gradual course changes, mimicking an intelligence-gathering vessel mapping the subsea cable routes or turbine positions.

The goal is not physical damage but **strategic deception**: triggering a military or coast guard response that diverts security resources, creating diplomatic tension, or providing cover for actual illicit activity elsewhere in the North Sea.

### Why This Is Plausible

| Aspect | Justification |
|--------|---------------|
| **Location** | 51.720°N, 3.050°E —> adjacent to the Borssele III/IV wind farm site, approximately 22 km off the Zeeland coast |
| **MMSI** | 273451200 —> valid Russian Federation prefix (273), fictitious vessel number |
| **SOG = 3.2** | Low speed consistent with survey or loitering behavior; within normal range in the dataset |
| **Nav status = 0** | "Under way using engine" —> does not draw immediate attention the way "not under command" would; appears as routine traffic until the loitering pattern is noticed |
| **COG = 180°, Heading = 175°** | Heading south with slight drift —> consistent with a vessel conducting parallel survey lines across the wind farm zone |
| **ROT = 5.0** | Moderate turn rate indicating gradual course changes —> characteristic of a vessel systematically covering an area rather than transiting |
| **Post-Nord Stream context** | Since 2022, any vessel with a Russian flag loitering near North Sea energy infrastructure would trigger immediate security scrutiny |

### Potential Impact

- Coast guard and/or naval assets dispatched to investigate a vessel that does not exist, diverting resources from real security tasks
- Diplomatic tension between the Netherlands and Russia based on fabricated AIS evidence of a surveillance vessel
- If combined with real covert activity elsewhere, the spoofed vessel serves as a decoy, a classic misdirection tactic
- Erosion of trust in AIS-based maritime domain awareness for the entire southern North Sea region
- The scenario demonstrates that AIS spoofing can be used for **strategic deception**, not just navigational disruption

In [14]:
# Scenario 3: Spoofed vessel field values
scenario_3 = {
    'type': 1,
    'mmsi': '273451200',       # Russian MMSI prefix (273), fictitious vessel
    'status': 0,               # Under way using engine
    'turn': 5.0,               # Moderate turn, loitering/survey pattern
    'speed': 3.2,              # Low speed, survey-like behavior
    'accuracy': 1,             # High accuracy GPS
    'lon': 3.0500,             # Adjacent to Borssele III/IV wind farm
    'lat': 51.7200,            # Approximately 22 km off Zeeland coast
    'course': 180.0,           # Heading south, systematic survey line
    'heading': 175,            # Slight offset from COG, gradual maneuvering
    'second': 22,
    'maneuver': 0,
    'raim': 0,
}

encoded_3, decoded_3 = encode_and_verify(
    scenario_3, "Scenario 3: Suspicious vessel near Borssele wind farm"
)


  Scenario 3: Suspicious vessel near Borssele wind farm

  INPUT FIELD VALUES:
    MMSI:       273451200
    Lat:        51.72
    Lon:        3.05
    SOG:        3.2 knots
    COG:        180.0 degrees
    Heading:    175 degrees
    ROT:        5.0
    Nav Status: 0

  ENCODED NMEA SENTENCE:
    !AIVDO,1,1,,A,144j8h02hPP=uSPMV2h725Nd0000,0*55

  DECODED BACK:
    MMSI:       273451200
    Lat:        51.72
    Lon:        3.05
    SOG:        3.2 knots
    COG:        180.0 degrees
    Heading:    175 degrees
    ROT:        5.0
    Nav Status: 0

  ROUND-TRIP: PASS


In [15]:
### Geographic Context

# ![Scenario 3 — Overview](../data/processed/Scenario3_overview.png)

# ![Scenario 3 — Zoomed](../data/processed/Scenario3_zoomed.png)

---
## Bonus Scenario: Fake Aid-to-Navigation (AtoN) in IJmuiden Shipping Lane

### Threat Narrative

Aids-to-Navigation (AtoN) are critical maritime safety assets; buoys, lighthouses, and beacons; that broadcast AIS Type 21 messages to help vessels navigate safely. AIS-based **virtual AtoN** are particularly interesting from a spoofing perspective: they represent navigational markers that have no physical presence and exist only as AIS broadcasts. This means there is no physical object to cross-check against visually or via radar.

The port of **IJmuiden** serves as the access point to the North Sea Canal and the Port of Amsterdam, and is the operational base for servicing the Hollandse Kust Zuid wind farm. The approach to IJmuiden is marked with lateral fairway markers that vessels rely on for safe navigation.

In this scenario, an attacker spoofs an AIS Type 21 message placing a **fake virtual hazard marker** in the IJmuiden approach lane. Vessels relying on AIS-based navigation displays would see a navigational hazard that does not physically exist, potentially causing them to alter course unnecessarily. Unlike the vessel-based scenarios above, this demonstrates that AIS spoofing threats extend beyond position reports to the **navigational infrastructure itself**.

### Note on Implementation

AIS Type 21 messages have a different field structure than Type 1/2/3 position reports —> they include fields for AtoN type, name, and dimensions rather than vessel movement data like SOG or COG. The `pyais` `encode_dict` function may or may not support Type 21 encoding. This scenario attempts to encode the message; if encoding is not supported, the field values are documented as a conceptual demonstration with the technical implementation deferred to semester two.

### Conceptual Field Values

| Field | Value | Justification |
|-------|-------|---------------|
| **Type** | 21 | AIS Message Type 21 —> Aid-to-Navigation Report |
| **MMSI** | 992446001 | Dutch AtoN MMSI: prefix 992 + MID 244 (Netherlands) + sequence |
| **Lat** | 52.4700°N | IJmuiden approach lane |
| **Lon** | 4.4500°E | In the shipping lane, not on shore |
| **Name** | HAZARD_MARK_A1 | Generic hazard marker designation |

### Potential Impact

- Vessels alter course to avoid a non-existent hazard, potentially entering shallow water or conflicting with other traffic
- Autonomous vessels following AIS-based route planning may automatically reroute
- The attack demonstrates that spoofing extends beyond vessel impersonation to manipulation of the navigational environment itself
- Virtual AtoN are inherently unverifiable by radar or visual observation, making this attack harder to detect than vessel spoofing

In [16]:
# Bonus: Fake AtoN field values
# Type 21 AtoN encoding via pyais: attempt encoding, fall back to conceptual if unsupported

try:
    bonus_aton = {
        'type': 21,
        'mmsi': '992446001',       # Dutch AtoN MMSI (prefix 992 + MID 244)
        'accuracy': 1,
        'lon': 4.4500,             # IJmuiden approach lane
        'lat': 52.4700,
        'shipname': 'HAZARD_MARK_A1',
        'second': 30,
        'raim': 0,
    }

    encoded_aton = encode_dict(bonus_aton)
    print(f"\n{'='*70}")
    print(f"  Bonus: Fake AtoN: IJmuiden approach")
    print(f"{'='*70}")
    print(f"\n  ENCODED AtoN NMEA SENTENCE:")
    print(f"    {encoded_aton[0]}")

    decoded_aton = decode(*encoded_aton).asdict()
    print(f"\n  DECODED BACK:")
    print(f"    MMSI:  {decoded_aton.get('mmsi', 'N/A')}")
    print(f"    Lat:   {decoded_aton.get('lat', 'N/A')}")
    print(f"    Lon:   {decoded_aton.get('lon', 'N/A')}")
    print(f"    Name:  {decoded_aton.get('shipname', decoded_aton.get('name', 'N/A'))}")
    print(f"\n  ROUND-TRIP: AtoN encoding supported: structurally valid")

except Exception as e:
    print(f"\n{'='*70}")
    print(f"  Bonus: Fake AtoN: IJmuiden approach (conceptual)")
    print(f"{'='*70}")
    print(f"\n  AtoN (Type 21) encoding via encode_dict raised:")
    print(f"    {type(e).__name__}: {e}")
    print(f"\n  Type 21 encoding is not supported by the current pyais version.")
    print(f"  The field values above are documented as a conceptual demonstration.")
    print(f"  Full Type 21 encoding support is deferred to semester two.")


  Bonus: Fake AtoN: IJmuiden approach

  ENCODED AtoN NMEA SENTENCE:
    !AIVDO,1,1,,A,E>jN6<@00000000000000000000@:;nh?0hB000000?00000000000000000,4*58

  DECODED BACK:
    MMSI:  992446001
    Lat:   52.47
    Lon:   4.45
    Name:  

  ROUND-TRIP: AtoN encoding supported: structurally valid


In [17]:
### Geographic Context

# ![Bonus — Overview](../data/processed/Bonus_overview.png)

---
## Summary

This notebook demonstrated that **structurally valid and contextually plausible spoofed AIS messages** can be constructed for four distinct threat scenarios in Dutch North Sea waters, using only the validated `pyais` encoder and knowledge of real AIS traffic distributions.

| Scenario | Target | Threat Type | Key Insight |
|----------|--------|-------------|-------------|
| **1. Ghost Vessel** | Nieuwe Waterweg / Maeslantkering | Port disruption | A single spoofed vessel at the right chokepoint can halt all traffic to Europe's largest port |
| **2. Collision Threat** | Autonomous vessel in HKZ wind farm | Safety / collision risk | Autonomous vessels trusting AIS over radar may take dangerous evasive action in confined spaces |
| **3. Deceptive Presence** | Borssele wind farm zone | Strategic deception | AIS spoofing can fabricate geopolitical incidents and divert security resources |
| **Bonus. Fake AtoN** | IJmuiden shipping lane | Navigation manipulation | Spoofing extends beyond vessels to the navigational infrastructure itself; virtual AtoN are inherently unverifiable |

### Common Characteristics

All scenarios share the following properties that make them feasible:

1. **No authentication**: AIS has no mechanism to verify the identity or legitimacy of a transmitting station
2. **Valid field values**: Every field value used is within the ITU-R M.1371 specification and consistent with real traffic patterns observed in the EDA
3. **Valid NMEA encoding**: All messages pass round-trip encode/decode verification, producing structurally correct NMEA 0183 sentences
4. **Geographic plausibility**: All coordinates are verified to be in water, at locations where the claimed vessel type would realistically operate
5. **Low barrier to execution**: Constructing these messages requires only knowledge of the AIS protocol and access to an encoder, no specialized hardware or advanced skills

### Implications for Semester Two

These manual scenarios establish what a human with domain knowledge can construct. Semester two will investigate whether **Generative AI** (a GAN trained on AIS data and an LLM with few-shot prompting) can produce equally plausible messages **without** the domain knowledge demonstrated here, which would represent a significant threat amplification by lowering the barrier to entry for AIS spoofing attacks.